In [ ]:
from readability import Readability
from readability.exceptions import ReadabilityException

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
!pip install bert-score

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, re, glob, warnings, shutil
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
from collections import Counter

# --- readability ---
from readability import Readability
from readability.exceptions import ReadabilityException

# =============================================================================
# CONFIG
# =============================================================================
OUTPUTS_ROOT = "Outputs"               # where DeepSeek/, OpenAI/, Gemma/, Mistral/ live
PROMPT_ASSETS = "../XAI_python_scripts/prompt_assets"  # where shap_<id>.csv and lime_<id>.csv live

FEATURES = [
    'epre','pusch_snr','p_ue','ul_mcs','cqi','ul_bitrate',
    'dl_mcs','dl_retx','ul_tx','dl_tx','ul_retx','dl_bitrate','dl_err','ul_err'
]

# Canonical-feature matcher that tolerates optional time suffixes like _t-2
# Uses custom boundaries (no letter/number/underscore on either side)
_FEATURE_VARIANTS = {
    f: re.compile(
        rf"(?<![A-Za-z0-9_]){re.escape(f)}(?:_t-?\d+)?(?![A-Za-z0-9_])",
        flags=re.IGNORECASE
    )
    for f in FEATURES
}

TOPK_SHAP_TEXT = 10                 # how many SHAP/LIME features to verbalize in reference text
TOPK_K = 5                          # K for Top-K extraction and Jaccard@K

# Debug / logging
LOG_DEBUG = True                    # print candidate/ref pairs for every prompt
LOG_FILE = "eval_debug.txt"         # also append to this file (set to None to disable)
CAND_SNIPPET_CHARS = 500            # limit candidate text in debug print/log to this many characters

# Readability preprocessing: treat each line as a sentence
PREPROCESS_READABILITY = True

# =============================================================================
# UTILITIES
# =============================================================================
def mean_without_outliers(values, z_thresh=2.0):
    """
    Compute mean excluding statistical outliers.
    Removes values more than z_thresh standard deviations from the mean.
    Returns np.nan if all values are filtered out.
    """
    arr = np.array(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return np.nan
    z_scores = np.abs((arr - np.mean(arr)) / np.std(arr, ddof=0)) if np.std(arr, ddof=0) > 0 else np.zeros_like(arr)
    filtered = arr[z_scores < z_thresh]
    return np.mean(filtered) if len(filtered) > 0 else np.nan

def normalize_feature_aliases(text: str) -> str:
    """
    Map common prose/variant phrasings to canonical FEATURE tokens so we can count them.
    Works on lowercase text. Extend with more lines if you see new phrasings.
    """
    s = text.lower()

    # --- exact phrases -> canonical tokens ---
    s = re.sub(r"\bchannel\s+quality\s+indicator\b", "cqi", s)

    # TX
    s = re.sub(r"\bul[\s\-]*tx\b", "ul_tx", s)
    s = re.sub(r"\bdl[\s\-]*tx\b", "dl_tx", s)

    # MCS
    s = re.sub(r"\bul[\s\-]*mcs\b", "ul_mcs", s)
    s = re.sub(r"\bdl[\s\-]*mcs\b", "dl_mcs", s)

    # Bitrate / bit rate / throughput (UL & DL)
    s = re.sub(r"\bul[\s\-]*bit\s*[\s\-]*rate\b", "ul_bitrate", s)
    s = re.sub(r"\bul[\s\-]*bitrate\b", "ul_bitrate", s)
    s = re.sub(r"\buplink\s+bit\s*[\s\-]*rate\b", "ul_bitrate", s)
    s = re.sub(r"\buplink\s+bitrate\b", "ul_bitrate", s)
    s = re.sub(r"\buplink\s+throughput\b", "ul_bitrate", s)

    s = re.sub(r"\bdl[\s\-]*bit\s*[\s\-]*rate\b", "dl_bitrate", s)
    s = re.sub(r"\bdl[\s\-]*bitrate\b", "dl_bitrate", s)
    s = re.sub(r"\bdownlink\s+bit\s*[\s\-]*rate\b", "dl_bitrate", s)
    s = re.sub(r"\bdownlink\s+bitrate\b", "dl_bitrate", s)
    s = re.sub(r"\bdownlink\s+throughput\b", "dl_bitrate", s)

    # Retransmissions / retx (UL & DL)
    s = re.sub(r"\bul[\s\-]*(?:retransmissions?|re\-?tx)\b", "ul_retx", s)
    s = re.sub(r"\bdl[\s\-]*(?:retransmissions?|re\-?tx)\b", "dl_retx", s)

    # Errors (UL & DL)
    s = re.sub(r"\buplink\s+errors?\b", "ul_err", s)
    s = re.sub(r"\bul[\s\-]*errors?\b", "ul_err", s)
    s = re.sub(r"\bdownlink\s+errors?\b", "dl_err", s)
    s = re.sub(r"\bdl[\s\-]*errors?\b", "dl_err", s)

    # PUSCH SNR (with/without underscore)
    s = re.sub(r"\bpusch[\s\-]*snr\b", "pusch_snr", s)

    # (EPRE, p_ue already match when lowercased)
    return s


def _read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

_num_tail = re.compile(r"(\d+)(?!.*\d)")
def _get_id_from_filename(fname: str) -> int:
    m = _num_tail.search(fname)
    return int(m.group(1)) if m else -1

def preprocess_for_readability(text: str) -> str:
    """
    Lightly normalize texts for readability:
      - remove bullet markers (*, -, •, – , —) at line starts,
      - drop stray asterisks used for emphasis,
      - collapse long dash separators,
      - convert trailing ':' at end-of-line to '.',
      - join non-empty lines into sentences.
    NOTE: This is ONLY used for readability metrics; the raw text is used elsewhere.
    """
    if not PREPROCESS_READABILITY:
        return text

    # Normalize newlines
    s = text.replace("\r\n", "\n").replace("\r", "\n")

    # 1) Remove bullet markers at the start of lines: *, -, •, en/em dash
    s = re.sub(r"(?m)^\s*[\*\-\u2022\u2013\u2014]\s+", "", s)

    # 2) Remove stray asterisks anywhere (markdown emphasis)
    s = s.replace("*", "")

    # 3) Collapse long dash separators (---, ———, etc.) to a single space
    s = re.sub(r"[—–\-]{2,}", " ", s)

    # 4) Convert a trailing colon at end-of-line into a period (avoid breaking times like 10:30)
    s = re.sub(r":\s*(\n|$)", r".\1", s)

    # 5) Strip other decorative bullets if present
    s = s.translate(str.maketrans({c: "" for c in "•▪►◦❖"}))

    # 6) Split -> keep non-empty lines -> join as sentences
    lines = [l.strip() for l in s.splitlines() if l.strip()]
    if not lines:
        return ""
    for l in lines:
        if not re.search(r"[.!?]$", l):
            l += "."

    out = ". ".join(lines)
    if not out.endswith("."):
        out += "."
    # Normalize whitespace
    out = re.sub(r"\s+", " ", out).strip()
    return out


def collect_outputs(model_root: str) -> Dict[str, List[Tuple[int,str]]]:
    """
    Return dict {mode: [(prompt_id, file), ...]} for zero_shot* and few_shot* subfolders.
    """
    modes = {}
    for sub in os.listdir(model_root):
        subpath = os.path.join(model_root, sub)
        if not os.path.isdir(subpath):
            continue
        if sub.startswith("zero_shot") or sub.startswith("few_shot"):
            files = sorted(
                glob.glob(os.path.join(subpath, "prompt_*.txt")),
                key=lambda p: _get_id_from_filename(os.path.basename(p))
            )
            items = []
            for p in files:
                pid = _get_id_from_filename(os.path.basename(p))
                if pid >= 0:
                    items.append((pid, p))
            if items:
                modes[sub] = items
    return modes

def compute_readability(text: str) -> Tuple[float, float, str]:
    """
    Returns (FleschReadingEase_score, GunningFog_score, GunningFog_grade_label).
    On error, returns (NaN, NaN, "").
    """
    try:
        r = Readability(preprocess_for_readability(text))
        fre  = float(getattr(r.flesch(), "score", np.nan))
        fog  = r.gunning_fog()
        fog_score = float(getattr(fog, "score", np.nan))
        fog_grade_label = getattr(fog, "grade_level", None)
        fog_grade_label = "" if fog_grade_label is None else str(fog_grade_label)
        return fre, fog_score, fog_grade_label
    except ReadabilityException:
        return np.nan, np.nan, ""
    except Exception:
        return np.nan, np.nan, ""


def _read_feature_ranking_csv(path: str) -> pd.Series:
    """
    Helper: read a CSV where columns are the FEATURES and rows are time steps;
    return a pd.Series of mean absolute values per feature, sorted desc.
    Falls back to first len(FEATURES) columns if header mismatch.
    """
    df = pd.read_csv(path, index_col=0)
    try:
        vals = df[FEATURES].abs().mean(axis=0).sort_values(ascending=False)
    except Exception:
        vals = df.iloc[:, :len(FEATURES)].abs().mean(axis=0).sort_values(ascending=False)
        vals.index = FEATURES[:len(vals)]
    return vals

def build_reference_text_from_shap(prompt_id: int) -> str:
    """
    Create a short textual reference from SHAP: list of top features as a sentence.
    """
    shap_csv = os.path.join(PROMPT_ASSETS, f"shap_{prompt_id}.csv")
    if not os.path.exists(shap_csv):
        return ""
    vals = _read_feature_ranking_csv(shap_csv)
    top_feats = list(vals.head(TOPK_SHAP_TEXT).index)
    return "Reference explanation (from SHAP): top features are " + ", ".join(top_feats) + "."

# Regex to find canonical feature names in arbitrary text, case-insensitive
_feat_regex = re.compile(r"\b(" + "|".join(map(re.escape, FEATURES)) + r")\b", flags=re.IGNORECASE)

def build_reference_text_from_lime(prompt_id: int) -> str:
    """
    Read PROMPT_ASSETS/lime_<id>.csv (columns like 'Feature', 'Contribution'),
    map rule-like strings (e.g., '0.01 < cqi_t-2 <= 0.54') to canonical base
    FEATURES ('cqi'), aggregate mean |contribution| per base feature, and
    return a sentence listing the top features.
    """
    lime_csv = os.path.join(PROMPT_ASSETS, f"lime_{prompt_id}.csv")
    if not os.path.exists(lime_csv):
        return ""

    df = pd.read_csv(lime_csv)

    # Column detection
    feat_col = next((c for c in ["Feature", "feature", "Features", "features", "Name", "name"] if c in df.columns), None)
    if feat_col is None:
        feat_col = df.columns[0]
    contrib_col = next((c for c in ["Contribution", "contribution", "Weight", "weight", "Value", "value"] if c in df.columns), None)
    if contrib_col is None:
        contrib_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]

    # Coerce numeric contribution and drop bad rows
    df[contrib_col] = pd.to_numeric(df[contrib_col], errors="coerce")
    df = df.dropna(subset=[contrib_col])

    # Map each row's 'Feature' string to a canonical base feature
    def _map_to_base(s: str) -> str:
        if not isinstance(s, str):
            return ""
        for base, pat in _FEATURE_VARIANTS.items():
            if pat.search(s):
                return base
        return ""

    df["_base_feat"] = df[feat_col].apply(_map_to_base)
    df = df[df["_base_feat"] != ""]
    if df.empty:
        return ""

    # Aggregate by base feature using mean absolute contribution
    agg = (
        df.assign(abs_contrib=df[contrib_col].abs())
          .groupby("_base_feat", as_index=True)["abs_contrib"]
          .mean()
    )

    # Stable sort: by value desc, then by FEATURES order
    order = {f: i for i, f in enumerate(FEATURES)}
    agg = agg.sort_values(ascending=False)
    agg = agg.loc[sorted(agg.index, key=lambda f: (-agg.loc[f], order.get(f, 10**9)))]

    top_feats = list(agg.head(TOPK_SHAP_TEXT).index)
    return "Reference explanation (from LIME): top features are " + ", ".join(top_feats) + "."

def extract_topk_features(text: str, k: int = TOPK_K) -> List[str]:
    """
    Count mentions of canonical FEATURES in the LLM text, robust to:
      - no-underscore forms like 'ul bitrate', 'dl tx',
      - prose aliases ('channel quality indicator' -> cqi),
      - optional time suffixes (feature_t-2),
      - case variations.
    Returns Top-K features by frequency; ties broken by FEATURES order.
    """
    s = normalize_feature_aliases(text)

    # Count by canonical feature using tolerant patterns (_FEATURE_VARIANTS handles optional _t-2)
    counts = {f: 0 for f in FEATURES}
    for f, pat in _FEATURE_VARIANTS.items():
        counts[f] = len(pat.findall(s))

    # Rank: frequency desc, then FEATURES order for stability
    order = {f: i for i, f in enumerate(FEATURES)}
    ranked = sorted(counts.items(), key=lambda kv: (-kv[1], order[kv[0]]))
    return [f for f, c in ranked if c > 0][:k]


def jaccard_at_k(pred: List[str], ref: List[str], k: int = TOPK_K) -> float:
    """Compute Jaccard@k between predicted top-k and reference top-k features."""
    pred_k = set(pred[:k])
    ref_k  = set(ref[:k])
    if not (pred_k or ref_k):
        return np.nan
    return len(pred_k & ref_k) / len(pred_k | ref_k)

def jaccard_sets(pred_list: List[str], ref_list: List[str]) -> float:
    """
    Jaccard between the model's Top-K (pred_list[:TOPK_K]) and an arbitrary
    reference set (e.g., union of SHAP/LIME Top-5). Returns NaN if both empty.
    """
    A = set(pred_list[:TOPK_K])
    B = set(ref_list)  # can be any size (e.g., up to 10 for union)
    if not (A or B):
        return np.nan
    U = A | B
    return len(A & B) / len(U) if U else np.nan

# =============================================================================
# MAIN
# =============================================================================
def evaluate_outputs():
    # Validate root
    if not os.path.isdir(OUTPUTS_ROOT):
        print(f"⚠️  OUTPUTS_ROOT not found: {os.path.abspath(OUTPUTS_ROOT)}")
        return

    models = [d for d in os.listdir(OUTPUTS_ROOT) if os.path.isdir(os.path.join(OUTPUTS_ROOT, d))]
    if not models:
        print(f"⚠️  No model folders found under {os.path.abspath(OUTPUTS_ROOT)}")
        return

    # Clear debug log
    if LOG_FILE:
        try:
            with open(LOG_FILE, "w", encoding="utf-8") as _:
                pass
        except Exception:
            pass

    rows = []

    for model in sorted(models):
        model_root = os.path.join(OUTPUTS_ROOT, model)
        modes = collect_outputs(model_root)

        grouped = {"No-Shot": [], "Few-Shot": []}
        for sub, items in modes.items():
            if sub.startswith("zero_shot"):
                grouped["No-Shot"].extend(items)
            elif sub.startswith("few_shot"):
                grouped["Few-Shot"].extend(items)

        for mode_name, items in grouped.items():
            if not items:
                continue

            texts = []
            fre_scores, gf_scores = [], []
            fog_grade_labels = []

            jac_scores_shap, jac_scores_lime = [], []
            jac_scores_union = []
            n_shap_refs, n_lime_refs = 0, 0

            for pid, path in items:
                txt = _read_text(path)
                texts.append(txt)

                # Readability
                fre, gf, fog_label = compute_readability(txt)
                fre_scores.append(fre); gf_scores.append(gf)
                fog_grade_labels.append(fog_label)

                # Predicted top-k from LLM text
                pred_topk = extract_topk_features(txt, k=TOPK_K)

                # Build SHAP reference (text + features)
                ref_text_shap = build_reference_text_from_shap(pid)
                ref_feats_shap = [m.lower() for m in _feat_regex.findall(ref_text_shap)] if ref_text_shap.strip() else []
                if ref_feats_shap:
                    n_shap_refs += 1
                ref_topk_shap = ref_feats_shap[:TOPK_K]
                jac_scores_shap.append(jaccard_at_k(pred_topk, ref_topk_shap, k=TOPK_K))

                # Build LIME reference (text + features)
                ref_text_lime = build_reference_text_from_lime(pid)
                ref_feats_lime = [m.lower() for m in _feat_regex.findall(ref_text_lime)] if ref_text_lime.strip() else []
                if ref_feats_lime:
                    n_lime_refs += 1
                ref_topk_lime = ref_feats_lime[:TOPK_K]
                jac_scores_lime.append(jaccard_at_k(pred_topk, ref_topk_lime, k=TOPK_K))

                # Union of SHAP and LIME Top-5 (optimistic / recall-friendly)
                ref_union = list(set(ref_topk_shap) | set(ref_topk_lime))
                jac_scores_union.append(jaccard_sets(pred_topk, ref_union))

                # ---- DEBUG PRINT / LOG -------------------------------------------------
                if LOG_DEBUG:
                    header = f"[{model} | {mode_name} | prompt_{pid}]"
                    cand_snip = txt.strip()[:CAND_SNIPPET_CHARS].replace("\n", " ")
                    block = (
                        f"\n{header}\n"
                        f"Candidate (model output):\n{cand_snip}\n\n"
                        f"SHAP Reference:\n{ref_text_shap.strip()}\n"
                        f"LIME Reference:\n{ref_text_lime.strip()}\n\n"
                        f"Pred Top-{TOPK_K}: {pred_topk}\n"
                        f"SHAP Top-{TOPK_K}: {ref_topk_shap} | J@{TOPK_K}={np.nan if np.isnan(jac_scores_shap[-1]) else round(jac_scores_shap[-1],3)}\n"
                        f"LIME Top-{TOPK_K}: {ref_topk_lime} | J@{TOPK_K}={np.nan if np.isnan(jac_scores_lime[-1]) else round(jac_scores_lime[-1],3)}\n"
                        f"UNION Top-{len(ref_union)}: {sorted(ref_union)} | J_union={np.nan if np.isnan(jac_scores_union[-1]) else round(jac_scores_union[-1],3)}\n"
                        + ("-"*80) + "\n"
                    )
                    print(block)
                    if LOG_FILE:
                        try:
                            with open(LOG_FILE, "a", encoding="utf-8") as log:
                                log.write(block)
                        except Exception:
                            pass
                # -----------------------------------------------------------------------

            # Aggregate Jaccard
            jac_at5_shap = float(np.nanmean(jac_scores_shap)) if jac_scores_shap else np.nan
            jac_at5_lime = float(np.nanmean(jac_scores_lime)) if jac_scores_lime else np.nan
            jac_union_mean = float(np.nanmean(jac_scores_union)) if jac_scores_union else np.nan

            # Choose a single label per (Model, Mode): most common Fog grade label
            valid_labels = [g for g in fog_grade_labels if g]
            fog_grade_mode = Counter(valid_labels).most_common(1)[0][0] if valid_labels else ""

            # Compute average (excluding outliers) and also track min/max readability
            # Compute average (excluding outliers) and also track min/max readability
            flesch_mean = float(mean_without_outliers(fre_scores)) if fre_scores else np.nan
            gunning_mean = float(mean_without_outliers(gf_scores)) if gf_scores else np.nan
            
            # Identify min/max + their corresponding prompt IDs
            if fre_scores:
                flesch_min_idx = int(np.nanargmin(fre_scores))
                flesch_max_idx = int(np.nanargmax(fre_scores))
                flesch_min_val = float(fre_scores[flesch_min_idx])
                flesch_max_val = float(fre_scores[flesch_max_idx])
                flesch_min_pid = items[flesch_min_idx][0]  # prompt ID
                flesch_max_pid = items[flesch_max_idx][0]
            else:
                flesch_min_val = flesch_max_val = flesch_min_pid = flesch_max_pid = np.nan
            
            if gf_scores:
                gunning_min_idx = int(np.nanargmin(gf_scores))
                gunning_max_idx = int(np.nanargmax(gf_scores))
                gunning_min_val = float(gf_scores[gunning_min_idx])
                gunning_max_val = float(gf_scores[gunning_max_idx])
                gunning_min_pid = items[gunning_min_idx][0]
                gunning_max_pid = items[gunning_max_idx][0]
            else:
                gunning_min_val = gunning_max_val = gunning_min_pid = gunning_max_pid = np.nan
            
            row = {
                "Model": model,
                "Mode": mode_name,
            
                # Mean (outlier-filtered)
                "FleschReadingEase": flesch_mean,
                "GunningFog": gunning_mean,
            
                # Min/max + which prompt
                "FleschReadingEase_min": flesch_min_val,
                "FleschReadingEase_max": flesch_max_val,
                "FleschReadingEase_min_prompt": flesch_min_pid,
                "FleschReadingEase_max_prompt": flesch_max_pid,
            
                "GunningFog_min": gunning_min_val,
                "GunningFog_max": gunning_max_val,
                "GunningFog_min_prompt": gunning_min_pid,
                "GunningFog_max_prompt": gunning_max_pid,
            
                "FogGradeLabel": fog_grade_mode,
                "Jaccard@5_SHAP": jac_at5_shap,
                "Jaccard@5_LIME": jac_at5_lime,
                "Jaccard@Union": jac_union_mean,
                "N_outputs": len(items),
                "N_SHAP_refs": n_shap_refs,
                "N_LIME_refs": n_lime_refs,
            }
            rows.append(row)


    # =========================
    # Pretty print the summary
    # =========================
    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠️  No results to report (no outputs discovered).")
        return
    df = df.sort_values(["Model", "Mode"])

    for c in ["FleschReadingEase", "GunningFog",
              "Jaccard@5_SHAP", "Jaccard@5_LIME", "Jaccard@Union"]:
        if c in df.columns:
            df[c] = df[c].astype(float).round(3)

    ordered_cols = [
        "Model", "Mode",
        "FleschReadingEase", "FleschReadingEase_min", "FleschReadingEase_max",
        "FleschReadingEase_min_prompt", "FleschReadingEase_max_prompt",
        "GunningFog", "GunningFog_min", "GunningFog_max",
        "GunningFog_min_prompt", "GunningFog_max_prompt",
        "FogGradeLabel",
        "Jaccard@5_SHAP", "Jaccard@5_LIME", "Jaccard@Union",
        "N_outputs", "N_SHAP_refs", "N_LIME_refs"
    ]


    df = df[[c for c in ordered_cols if c in df.columns]]

    # Truncate wide text columns to fit terminal width
    term_width = shutil.get_terminal_size((120, 20)).columns
    max_col_width = max(15, term_width // max(1, len(df.columns)))
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(
            lambda x: (x[:max_col_width-1] + "…") if isinstance(x, str) and len(x) > max_col_width else x
        )

    # Pretty print with tabulate if available; fallback to markdown
    try:
        from tabulate import tabulate
        try:
            print(tabulate(df, headers="keys", tablefmt="fancy_grid", showindex=False,
                           numalign="right", maxcolwidths=[max_col_width]*len(df.columns)))
        except TypeError:
            # older tabulate without maxcolwidths
            print(tabulate(df, headers="keys", tablefmt="fancy_grid", showindex=False, numalign="right"))
    except Exception:
        print(df.to_markdown(index=False))

    # Save CSV for LaTeX table creation
    out_csv = "llm_eval_summary.csv"
    df.to_csv(out_csv, index=False)
    print(f"\n✅ Saved summary to {out_csv}")
    if LOG_FILE:
        print(f"📝 Debug log written to: {LOG_FILE}")

if __name__ == "__main__":
    evaluate_outputs()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, re, glob, warnings, shutil
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd

from bert_score import score as bertscore

# =========================
# CONFIG
# =========================
OUTPUTS_ROOT = "Outputs"          # where DeepSeek/, OpenAI/, Gemma/, Mistral/ live
HUMAN_REFS_DIR = "Human_Refs"     # <-- your folder with human-written refs

BERT_MODEL = "roberta-large"
BERT_LANG = "en"
BERT_USE_IDF = True               # stabilize; focus on content words
BERT_RESCALE_BASELINE = False     # keep raw 0..1-ish similarities (no negatives)

LOG_DEBUG = True                  # print a few candidate/reference pairs
CAND_SNIPPET_CHARS = 300

# =========================
# HELPERS
# =========================
_num_tail = re.compile(r"(\d+)(?!.*\d)")

def _get_id_from_filename(fname: str) -> int:
    """Extract trailing integer, e.g., prompt_12.txt -> 12."""
    m = _num_tail.search(fname)
    return int(m.group(1)) if m else -1

def _read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def collect_outputs(model_root: str) -> Dict[str, List[Tuple[int,str]]]:
    """
    Return dict {mode: [(prompt_id, file), ...]} for zero_shot* and few_shot* subfolders.
    """
    modes = {}
    for sub in os.listdir(model_root):
        subpath = os.path.join(model_root, sub)
        if not os.path.isdir(subpath):
            continue
        if sub.startswith("zero_shot") or sub.startswith("few_shot"):
            files = sorted(
                glob.glob(os.path.join(subpath, "prompt_*.txt")),
                key=lambda p: _get_id_from_filename(os.path.basename(p))
            )
            items = []
            for p in files:
                pid = _get_id_from_filename(os.path.basename(p))
                if pid >= 0:
                    items.append((pid, p))
            if items:
                modes[sub] = items
    return modes

def _find_human_ref_path(pid: int) -> str:
    """
    Try common filenames inside HUMAN_REFS_DIR:
      - human_ref_<id>.txt
      - prompt_<id>.txt
      - <id>.txt
    Fallback: any *.txt whose trailing number matches pid.
    """
    if not os.path.isdir(HUMAN_REFS_DIR):
        return ""
    candidates = [
        os.path.join(HUMAN_REFS_DIR, f"human_ref_{pid}.txt"),
        os.path.join(HUMAN_REFS_DIR, f"prompt_{pid}.txt"),
        os.path.join(HUMAN_REFS_DIR, f"{pid}.txt"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    for p in glob.glob(os.path.join(HUMAN_REFS_DIR, "*.txt")):
        if _get_id_from_filename(os.path.basename(p)) == pid:
            return p
    return ""

def load_human_reference(pid: int) -> str:
    p = _find_human_ref_path(pid)
    return _read_text(p) if p else ""

# =========================
# MAIN
# =========================
def evaluate_outputs():
    if not os.path.isdir(OUTPUTS_ROOT):
        print(f"⚠️  OUTPUTS_ROOT not found: {os.path.abspath(OUTPUTS_ROOT)}")
        return

    models = [d for d in os.listdir(OUTPUTS_ROOT) if os.path.isdir(os.path.join(OUTPUTS_ROOT, d))]
    if not models:
        print(f"⚠️  No model folders found under {os.path.abspath(OUTPUTS_ROOT)}")
        return

    rows = []

    for model in sorted(models):
        model_root = os.path.join(OUTPUTS_ROOT, model)
        modes = collect_outputs(model_root)

        # Canonical group names
        grouped = {"No-Shot": [], "Few-Shot": []}
        for sub, items in modes.items():
            if sub.startswith("zero_shot"):
                grouped["No-Shot"].extend(items)
            elif sub.startswith("few_shot"):
                grouped["Few-Shot"].extend(items)

        for mode_name, items in grouped.items():
            if not items:
                continue

            texts = []
            human_refs = []
            have_ref_mask = []

            for pid, path in items:
                txt = _read_text(path)
                texts.append(txt)

                ref_txt = load_human_reference(pid)
                human_refs.append(ref_txt)
                have_ref_mask.append(bool(ref_txt.strip()))

            # ----- BERTScore: LLM (full text) vs Human (full text) -----
            bs_f1 = np.nan
            cand = [t for t, ok in zip(texts, have_ref_mask) if ok]
            ref  = [r for r, ok in zip(human_refs, have_ref_mask) if ok]

            if cand:
                try:
                    _, _, F1 = bertscore(
                        cand, ref,
                        model_type=BERT_MODEL,
                        lang=BERT_LANG,
                        idf=BERT_USE_IDF,
                        rescale_with_baseline=BERT_RESCALE_BASELINE
                    )
                    bs_f1 = float(F1.mean().item())
                except Exception as e:
                    warnings.warn(f"BERTScore failed for {model} / {mode_name}: {e}")
                    bs_f1 = np.nan

            if LOG_DEBUG and cand:
                print("\n[BERTScore comparisons (LLM vs Human)]")
                for i in range(min(3, len(cand))):
                    print("cand:", cand[i][:CAND_SNIPPET_CHARS].replace("\n", " "))
                    print("ref :",  ref[i][:CAND_SNIPPET_CHARS].replace("\n", " "))
                    print("---")


            row = {
                "Model": model,
                "Mode": mode_name,
                "BERTScore_F1": bs_f1,
                "N_outputs": len(items),
                "N_refs_used": int(sum(have_ref_mask)),
            }
            rows.append(row)

    # =========================
    # Pretty print the summary
    # =========================
    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠️  No results to report (no outputs discovered).")
        return
    df = df.sort_values(["Model", "Mode"])

    if "BERTScore_F1" in df.columns:
        df["BERTScore_F1"] = df["BERTScore_F1"].astype(float).round(3)

    # Width-capped pretty table
    term_width = shutil.get_terminal_size((120, 20)).columns
    max_col_width = max(15, term_width // max(1, len(df.columns)))
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(
            lambda x: (x[:max_col_width-1] + "…") if isinstance(x, str) and len(x) > max_col_width else x
        )

    try:
        from tabulate import tabulate
        try:
            print(tabulate(df, headers="keys", tablefmt="fancy_grid", showindex=False,
                           numalign="right", maxcolwidths=[max_col_width]*len(df.columns)))
        except TypeError:
            print(tabulate(df, headers="keys", tablefmt="fancy_grid", showindex=False, numalign="right"))
    except Exception:
        print(df.to_markdown(index=False))

    out_csv = "llm_eval_bertscore_human_only.csv"
    df.to_csv(out_csv, index=False)
    print(f"\n✅ Saved summary to {out_csv}")

if __name__ == "__main__":
    evaluate_outputs()


In [ ]:
# ---- CONFIG ----
OUTPUTS_ROOT = "Outputs"
MODELS = ["OpenAI", "OpenAI_reasoning", "DeepSeek", "DeepSeek_reasoning", "Gemma", "Mistral"]

# Folder names for the two runs per mode (matches your screenshot)
RUNS_MAP = {
    "No-Shot": ("zero_shot", "zero_shot_run_2"),
    "Few-Shot": ("few_shot", "few_shot_run_2"),
}

# BERTScore settings
BERT_MODEL = "roberta-large"
BERT_LANG = "en"
BERT_USE_IDF = True
BERT_RESCALE_BASELINE = False

LOG_DEBUG = True
CAND_SNIP = 300


# ---- HELPERS ----
import os, re, glob, warnings
import numpy as np
import pandas as pd
from bert_score import score as bertscore

def _read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def _get_id_from_filename(fname: str) -> int:
    m = re.search(r"(\d+)(?!.*\d)", fname)
    return int(m.group(1)) if m else -1

def _load_prompt_texts(folder: str):
    """Return {prompt_id: text} for all prompt_*.txt in a folder."""
    texts = {}
    for p in glob.glob(os.path.join(folder, "prompt_*.txt")):
        pid = _get_id_from_filename(os.path.basename(p))
        if pid >= 0:
            texts[pid] = _read_text(p)
    return texts


# ---- MAIN ----
def compare_all_models_two_runs():
    summary_rows = []
    per_prompt_rows = []

    for model in MODELS:
        model_root = os.path.join(OUTPUTS_ROOT, model)
        if not os.path.isdir(model_root):
            print(f"⚠️  Skipping {model}: folder not found at {os.path.abspath(model_root)}")
            continue

        for mode, (run1_dir, run2_dir) in RUNS_MAP.items():
            dir1 = os.path.join(model_root, run1_dir)
            dir2 = os.path.join(model_root, run2_dir)

            if not os.path.isdir(dir1) or not os.path.isdir(dir2):
                print(f"⚠️  {model} | {mode}: missing '{run1_dir}' or '{run2_dir}'")
                summary_rows.append({
                    "Model": model, "Mode": mode,
                    "BERTScore_P_mean": np.nan,
                    "BERTScore_R_mean": np.nan,
                    "BERTScore_F1_mean": np.nan,
                    "N_pairs_used": 0
                })
                continue

            texts1 = _load_prompt_texts(dir1)   # Run 1
            texts2 = _load_prompt_texts(dir2)   # Run 2
            common = sorted(set(texts1) & set(texts2))
            if not common:
                print(f"⚠️  {model} | {mode}: no overlapping prompt IDs.")
                summary_rows.append({
                    "Model": model, "Mode": mode,
                    "BERTScore_P_mean": np.nan,
                    "BERTScore_R_mean": np.nan,
                    "BERTScore_F1_mean": np.nan,
                    "N_pairs_used": 0
                })
                continue

            candA = [texts1[pid] for pid in common]
            candB = [texts2[pid] for pid in common]

            try:
                P, R, F1 = bertscore(
                    candA, candB,
                    model_type=BERT_MODEL,
                    lang=BERT_LANG,
                    idf=BERT_USE_IDF,
                    rescale_with_baseline=BERT_RESCALE_BASELINE
                )
                P = P.cpu().numpy(); R = R.cpu().numpy(); F1 = F1.cpu().numpy()
            except Exception as e:
                warnings.warn(f"BERTScore failed for {model} | {mode}: {e}")
                P = R = F1 = np.array([])

            if P.size:
                summary_rows.append({
                    "Model": model,
                    "Mode": mode,
                    "BERTScore_P_mean": float(np.mean(P)),
                    "BERTScore_R_mean": float(np.mean(R)),
                    "BERTScore_F1_mean": float(np.mean(F1)),
                    "N_pairs_used": len(common),
                })
                for pid, p, r, f in zip(common, P, R, F1):
                    per_prompt_rows.append({
                        "Model": model,
                        "Mode": mode,
                        "PromptID": pid,
                        "BERTScore_P": float(p),
                        "BERTScore_R": float(r),
                        "BERTScore_F1": float(f),
                    })

                if LOG_DEBUG:
                    print(f"\n[BERTScore Run1 vs Run2] {model} | {mode} | pairs={len(common)}")
                    for pid, a, b in list(zip(common, candA, candB))[:3]:
                        print(f"prompt_{pid} | Run1:", a[:CAND_SNIP].replace("\n", " "))
                        print(f"prompt_{pid} | Run2:", b[:CAND_SNIP].replace("\n", " "))
                        print("---")
            else:
                summary_rows.append({
                    "Model": model, "Mode": mode,
                    "BERTScore_P_mean": np.nan,
                    "BERTScore_R_mean": np.nan,
                    "BERTScore_F1_mean": np.nan,
                    "N_pairs_used": 0
                })

    # ---- save CSVs ----
    if not summary_rows:
        print("⚠️  No comparisons to report.")
        return

    summary_df = pd.DataFrame(summary_rows).sort_values(["Model", "Mode"])
    for c in ["BERTScore_P_mean", "BERTScore_R_mean", "BERTScore_F1_mean"]:
        if c in summary_df:
            summary_df[c] = summary_df[c].astype(float).round(3)

    per_prompt_df = pd.DataFrame(per_prompt_rows).sort_values(["Model", "Mode", "PromptID"])
    for c in ["BERTScore_P", "BERTScore_R", "BERTScore_F1"]:
        if c in per_prompt_df:
            per_prompt_df[c] = per_prompt_df[c].astype(float).round(3)

    summary_csv = "all_models_run1_vs_run2_bertscore_summary.csv"
    byprompt_csv = "all_models_run1_vs_run2_bertscore_by_prompt.csv"
    summary_df.to_csv(summary_csv, index=False)
    per_prompt_df.to_csv(byprompt_csv, index=False)

    print("\n✅ Saved:")
    print(f" - {summary_csv}")
    print(f" - {byprompt_csv}")


# Run it
if __name__ == "__main__":
    compare_all_models_two_runs()
